# 4.3 Relationships in Data — Tutorial

**Primary outcome:** Quantify and model relationships between variables using correlation coefficients, simple linear regression, and multiple linear regression, with diagnostic checks throughout.

**Estimated time:** ~60–90 minutes

---

## Learning Objectives

By the end of this tutorial, you will be able to:

- Identify and describe the four main types of relationships between variables
- Calculate and interpret Pearson and Spearman correlation coefficients
- Build and evaluate a simple linear regression model
- Perform residual analysis to validate model assumptions
- Build a multiple linear regression model with categorical features
- Check for multicollinearity using Variance Inflation Factor (VIF)
- Make predictions using a trained regression model

---

## Datasets

- **`tips`** (seaborn) — restaurant bill and tip data; used for correlation and simple regression
- **`mpg`** (seaborn) — automobile fuel efficiency data; used for multiple regression

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import pearsonr, spearmanr, probplot

# Machine learning
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Multicollinearity check
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Reproducibility and aesthetics
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("All libraries loaded successfully.")

---

## 1. Types of Relationships

Before we measure relationships numerically, it helps to recognise them visually. There are four fundamental patterns:

| Pattern | What it looks like | Pearson r |
|---|---|---|
| **Linear positive** | Points trend upward in a straight line | Close to +1 |
| **Linear negative** | Points trend downward in a straight line | Close to −1 |
| **Non-linear** | Curved or U-shaped pattern | Near 0 despite a real relationship |
| **No relationship** | Random scatter, no visible pattern | Near 0 |

> **Key insight:** A Pearson r near 0 does *not* always mean no relationship — it only means no *linear* relationship. Non-linear patterns can still be strong.

Let's generate synthetic data to illustrate all four cases.

In [ ]:
np.random.seed(42)
n = 100
x = np.linspace(-3, 3, n)

# Four relationship types
y_pos  = 2 * x + np.random.normal(0, 0.8, n)          # Linear positive
y_neg  = -2 * x + np.random.normal(0, 0.8, n)         # Linear negative
y_nl   = x**2 + np.random.normal(0, 0.5, n)           # Non-linear (quadratic)
y_none = np.random.normal(0, 2, n)                     # No relationship

datasets = [
    (x, y_pos,  "Linear Positive"),
    (x, y_neg,  "Linear Negative"),
    (x, y_nl,   "Non-linear"),
    (x, y_none, "No Relationship"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (xi, yi, title) in zip(axes, datasets):
    r, _ = pearsonr(xi, yi)
    ax.scatter(xi, yi, alpha=0.6, edgecolors='white', linewidth=0.4)
    ax.set_title(f"{title}\nPearson r = {r:.2f}", fontsize=13, fontweight='bold')
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.suptitle("Four Types of Relationships Between Variables", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nNote the non-linear case: r is near 0 yet the relationship is very real!")

---

## 2. Correlation Analysis

Correlation quantifies the *strength* and *direction* of a linear relationship between two variables. The result is always between −1 and +1:

- **r = +1** — perfect positive linear relationship
- **r = 0** — no linear relationship
- **r = −1** — perfect negative linear relationship

We'll use the **`tips`** dataset — a classic restaurant dataset recording the total bill, tip amount, party size, day, and other features.

In [ ]:
# Load the tips dataset
tips = sns.load_dataset('tips')
print("Shape:", tips.shape)
print("\nFirst 5 rows:")
tips.head()

In [ ]:
# Full correlation matrix (numeric columns only)
numeric_tips = tips.select_dtypes(include='number')
corr_matrix = numeric_tips.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)
plt.title("Correlation Heatmap — Tips Dataset", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation matrix:")
print(corr_matrix.round(3))

### 2.1 Pearson Correlation

**Pearson r** measures the linear relationship between two *continuous, normally distributed* variables. It also gives you a **p-value** — the probability of observing a correlation this strong (or stronger) purely by chance if the true correlation were zero.

- **p < 0.05** → statistically significant (unlikely due to chance)
- **p ≥ 0.05** → not statistically significant

In [ ]:
# Pearson correlation between total_bill and tip
r, p = pearsonr(tips['total_bill'], tips['tip'])

print("=== Pearson Correlation: total_bill vs tip ===")
print(f"  r = {r:.4f}")
print(f"  p-value = {p:.2e}")
print()

# Interpret r strength
if abs(r) >= 0.7:
    strength = "strong"
elif abs(r) >= 0.4:
    strength = "moderate"
else:
    strength = "weak"

direction = "positive" if r > 0 else "negative"

print(f"Interpretation:")
print(f"  r = {r:.2f} indicates a {strength} {direction} linear relationship.")
print(f"  Customers who pay higher bills tend to leave higher tips.")
print()

if p < 0.001:
    print(f"  p = {p:.2e} — extremely significant (p < 0.001).")
    print(f"  This correlation is almost certainly not due to chance.")
elif p < 0.05:
    print(f"  p = {p:.4f} — statistically significant (p < 0.05).")
else:
    print(f"  p = {p:.4f} — not statistically significant.")

# Visual
plt.figure(figsize=(8, 5))
sns.scatterplot(data=tips, x='total_bill', y='tip', alpha=0.6)
plt.title(f"Total Bill vs Tip  (r = {r:.2f}, p = {p:.2e})", fontsize=13)
plt.xlabel("Total Bill ($)")
plt.ylabel("Tip ($)")
plt.tight_layout()
plt.show()

### 2.2 Spearman Correlation

**Spearman's rho (ρ)** measures the *monotonic* relationship between two variables — it works by ranking the data first, so it is:

- More robust to **outliers**
- Suitable when data is **not normally distributed** or is **ordinal**
- Captures non-linear (but still monotonic) relationships

### When to use Pearson vs Spearman

| | Pearson | Spearman |
|---|---|---|
| **Best for** | Normally distributed continuous data | Ordinal data, skewed data, or data with outliers |
| **Assumption** | Linear relationship | Monotonic relationship |
| **Sensitive to outliers** | Yes — outliers can distort r significantly | No — works on ranks |
| **When to choose** | Both variables roughly normal, no extreme outliers | When in doubt, or when Pearson assumptions don't hold |

In [ ]:
# Spearman correlation for total_bill vs tip
rho, p_sp = spearmanr(tips['total_bill'], tips['tip'])

print("=== Spearman Correlation: total_bill vs tip ===")
print(f"  rho = {rho:.4f}")
print(f"  p-value = {p_sp:.2e}")
print()
print("=== Comparison ===")
print(f"  Pearson r  = {r:.4f}")
print(f"  Spearman rho = {rho:.4f}")
print()
print("Interpretation:")
print("  Both metrics agree on a moderate-to-strong positive relationship.")
print("  Spearman is slightly lower here, suggesting a few large bills/tips")
print("  slightly inflate the Pearson result — worth checking for outliers.")

### 2.3 Correlation Does Not Imply Causation

> **Just because two variables are correlated does not mean one causes the other.**

A classic example: ice cream sales and drowning deaths are positively correlated — but ice cream does not cause drownings. Both are driven by a **confounding variable**: hot weather.

In our tips dataset, `total_bill` and `tip` are correlated, but the bill amount does not *cause* a specific tip. Party size, service quality, time of day, and individual habits all play a role. To make causal claims, we need **controlled experiments** or **causal inference methods** — not just correlation.

---

## 3. Simple Linear Regression

Correlation tells us the *strength* of a relationship, but regression lets us **quantify and predict** it. Simple linear regression fits a straight line:

$$\hat{y} = b_0 + b_1 x$$

- $b_0$ = **intercept** — the predicted value of y when x = 0
- $b_1$ = **slope** — how much y changes for every 1-unit increase in x

We will predict `tip` from `total_bill` using the tips dataset.

In [ ]:
# Quick visual before modelling
x_vals = tips['total_bill'].values
y_vals = tips['tip'].values

# Manual regression line using numpy polyfit
slope_np, intercept_np = np.polyfit(x_vals, y_vals, 1)
x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
y_line = slope_np * x_line + intercept_np

plt.figure(figsize=(9, 5))
plt.scatter(x_vals, y_vals, alpha=0.5, label='Observed data', edgecolors='white', linewidth=0.3)
plt.plot(x_line, y_line, color='crimson', linewidth=2, label=f'Regression line  (slope={slope_np:.3f})')
plt.title("Simple Linear Regression: Total Bill to Tip", fontsize=13, fontweight='bold')
plt.xlabel("Total Bill ($)")
plt.ylabel("Tip ($)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"numpy polyfit — slope: {slope_np:.4f}, intercept: {intercept_np:.4f}")

In [ ]:
# Fit with sklearn LinearRegression
X = tips[['total_bill']]   # 2-D array required by sklearn
y = tips['tip']

model = LinearRegression()
model.fit(X, y)

slope     = model.coef_[0]
intercept = model.intercept_
r2        = model.score(X, y)

print("=== Simple Linear Regression: tip ~ total_bill ===")
print(f"  Intercept (b0) = {intercept:.4f}")
print(f"  Slope     (b1) = {slope:.4f}")
print(f"  R-squared      = {r2:.4f}")
print()
print("Interpretation:")
print(f"  For every $1 increase in total bill, the tip increases by ${slope:.2f} on average.")
print(f"  When total_bill = $0, the predicted tip is ${intercept:.2f}")
print(f"  (the intercept is not meaningful here — nobody pays a $0 bill.)")
print()
print(f"  R-squared = {r2:.2f} means total_bill explains {r2*100:.1f}% of the variance in tip.")
print(f"  The remaining {(1-r2)*100:.1f}% is driven by other factors (service, mood, party size...).")

In [ ]:
# Scatter plot + regression line + residuals highlighted
y_pred = model.predict(X)
residuals = y - y_pred

fig, ax = plt.subplots(figsize=(10, 6))

# Draw residual lines first (so they sit behind the points)
for xi, yi, yp in zip(X['total_bill'], y, y_pred):
    ax.plot([xi, xi], [yi, yp], color='#aaaaaa', linewidth=0.6, zorder=1)

# Scatter and regression line
ax.scatter(X['total_bill'], y, alpha=0.6, zorder=2, label='Observed', edgecolors='white', linewidth=0.3)
ax.plot(x_line, slope * x_line + intercept, color='crimson', linewidth=2, zorder=3, label='Fitted line')

ax.set_title("Regression Line with Residuals (grey lines = errors)", fontsize=13, fontweight='bold')
ax.set_xlabel("Total Bill ($)")
ax.set_ylabel("Tip ($)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.4f}  (should be ~0 for OLS)")
print(f"Std of residuals: {residuals.std():.4f}")

---

## 4. Residual Analysis

A regression line gives us *fitted values* ($\hat{y}$). The **residuals** are the differences between what we observed and what we predicted:

$$e_i = y_i - \hat{y}_i$$

Residual analysis is a diagnostic check — it tells us whether our model's assumptions hold. There are three main plots:

| Plot | What to look for | Bad sign |
|---|---|---|
| **Residuals vs Fitted** | Random scatter around zero | Fan shape (heteroscedasticity) or a curve (non-linearity) |
| **Histogram of residuals** | Roughly bell-shaped, centred on 0 | Heavy skew or a bimodal distribution |
| **QQ plot** | Points follow the diagonal line | Systematic deviation from the line |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Residuals vs Fitted values ---
axes[0].scatter(y_pred, residuals, alpha=0.5, edgecolors='white', linewidth=0.3)
axes[0].axhline(0, color='crimson', linewidth=1.5, linestyle='--')
axes[0].set_xlabel("Fitted Values ($)")
axes[0].set_ylabel("Residuals ($)")
axes[0].set_title("Residuals vs Fitted", fontweight='bold')

# --- Plot 2: Histogram of residuals ---
axes[1].hist(residuals, bins=25, edgecolor='white', color='steelblue')
axes[1].axvline(0, color='crimson', linewidth=1.5, linestyle='--', label='Zero')
axes[1].set_xlabel("Residual Value ($)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Residuals", fontweight='bold')
axes[1].legend()

# --- Plot 3: QQ plot ---
probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title("QQ Plot of Residuals", fontweight='bold')

plt.suptitle("Residual Diagnostics — tip ~ total_bill", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Observations:")
print("  Residuals vs Fitted: slight fan shape suggests heteroscedasticity —")
print("    variance in tip increases as the bill increases. Not ideal but common.")
print("  Histogram: roughly bell-shaped but right-skewed — some unusually large tips.")
print("  QQ plot: points follow the diagonal reasonably in the middle,")
print("    with deviation at the upper tail (large positive residuals).")

---

## 5. Multiple Linear Regression

Simple regression uses *one* predictor. **Multiple linear regression** uses *several*:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \cdots + b_p x_p$$

Each coefficient $b_i$ represents the change in $y$ for a one-unit increase in $x_i$, **holding all other predictors constant**.

We will use the **`mpg`** dataset to predict fuel efficiency (`mpg`) from `horsepower`, `weight`, and `cylinders`.

In [ ]:
# Load the mpg dataset
mpg = sns.load_dataset('mpg')
print("Shape:", mpg.shape)
print("\nColumn dtypes:")
print(mpg.dtypes)
print("\nMissing values:")
print(mpg.isnull().sum())

In [ ]:
# Drop rows with any nulls (horsepower has a few)
mpg_clean = mpg.dropna().reset_index(drop=True)
print(f"Rows after dropping nulls: {len(mpg_clean)}  (dropped {len(mpg) - len(mpg_clean)} rows)")

# Correlation matrix of numeric features
numeric_mpg = mpg_clean.select_dtypes(include='number')
corr_mpg = numeric_mpg.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_mpg,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)
plt.title("Correlation Heatmap — MPG Dataset (numeric features)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey observations:")
print(f"  mpg vs horsepower : {corr_mpg.loc['mpg','horsepower']:.2f}  (strong negative)")
print(f"  mpg vs weight     : {corr_mpg.loc['mpg','weight']:.2f}  (strong negative)")
print(f"  mpg vs cylinders  : {corr_mpg.loc['mpg','cylinders']:.2f}  (strong negative)")
print("  Heavier, more powerful, higher-cylinder cars tend to get fewer miles per gallon.")

In [ ]:
# Encode categorical 'origin' as dummy variables
# 'origin' has values: 'usa', 'japan', 'europe'
origin_dummies = pd.get_dummies(mpg_clean['origin'], prefix='origin', drop_first=True)
print("Dummy columns created:", list(origin_dummies.columns))
print(origin_dummies.head())

In [ ]:
# Build feature matrix
features = ['horsepower', 'weight', 'cylinders']
X_mpg = pd.concat([mpg_clean[features], origin_dummies], axis=1)
y_mpg = mpg_clean['mpg']

print("Feature matrix shape:", X_mpg.shape)
print("Columns:", list(X_mpg.columns))

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_mpg, y_mpg, test_size=0.2, random_state=42
)
print(f"\nTrain size: {len(X_train)}  |  Test size: {len(X_test)}")

In [ ]:
# Fit multiple linear regression
mlr = LinearRegression()
mlr.fit(X_train, y_train)

r2_train = mlr.score(X_train, y_train)
r2_test  = mlr.score(X_test,  y_test)

# Print coefficients as a readable table
coeff_df = pd.DataFrame({
    'Feature': X_mpg.columns,
    'Coefficient': mlr.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"Intercept: {mlr.intercept_:.4f}")
print()
print(coeff_df.to_string(index=False))
print()
print("=== Model Performance ===")
print(f"  R-squared on training set : {r2_train:.4f}")
print(f"  R-squared on test set     : {r2_test:.4f}")
print()
print("Interpretation:")
hp_coef = coeff_df.loc[coeff_df['Feature']=='horsepower','Coefficient'].values[0]
wt_coef = coeff_df.loc[coeff_df['Feature']=='weight','Coefficient'].values[0]
print(f"  Each additional unit of horsepower is associated with a {abs(hp_coef):.4f} mpg decrease, all else equal.")
print(f"  Each additional lb of weight is associated with a {abs(wt_coef):.5f} mpg decrease, all else equal.")
print(f"  Train R-sq is close to Test R-sq — the model generalises well; no severe overfitting.")

---

## 6. Model Assumptions Check

Linear regression rests on **five core assumptions**:

| # | Assumption | What it means | How to check |
|---|---|---|---|
| 1 | **Linearity** | The relationship between predictors and outcome is linear | Residuals vs fitted plot — look for curves |
| 2 | **Independence** | Observations are independent of each other | Domain knowledge; Durbin-Watson test for time series |
| 3 | **Homoscedasticity** | Residuals have constant variance (not a fan/funnel shape) | Residuals vs fitted plot — look for expanding variance |
| 4 | **Normality** | Residuals are approximately normally distributed | QQ plot and histogram of residuals |
| 5 | **No multicollinearity** | Predictors are not highly correlated with each other | Variance Inflation Factor (VIF) |

### Variance Inflation Factor (VIF)

VIF measures how much the variance of a coefficient is inflated because of correlation with other predictors:

- **VIF = 1** — no correlation with other predictors
- **VIF 1–5** — moderate correlation, usually acceptable
- **VIF > 5–10** — high multicollinearity; consider removing one of the correlated predictors

Multicollinearity does not bias predictions, but it makes individual coefficients unreliable and hard to interpret.

In [ ]:
# VIF for each feature in the MLR model
from statsmodels.tools.tools import add_constant

X_vif = add_constant(X_mpg.astype(float))
vif_data = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})

# Exclude the constant row for display clarity
vif_data = vif_data[vif_data['Feature'] != 'const'].sort_values('VIF', ascending=False)

print("=== Variance Inflation Factors ===")
print(vif_data.to_string(index=False))
print()

high_vif = vif_data[vif_data['VIF'] > 5]
if len(high_vif) > 0:
    print("WARNING: Features with VIF > 5 (potential multicollinearity):")
    for _, row in high_vif.iterrows():
        print(f"  {row['Feature']}: VIF = {row['VIF']:.2f}")
    print()
    print("Interpretation:")
    print("  cylinders, horsepower, and weight are correlated with each other")
    print("  (powerful cars tend to have more cylinders and weigh more).")
    print("  High VIF means individual coefficients are less reliable —")
    print("  the model may still predict well, but coefficient interpretation is harder.")
    print("  Options: drop the highest-VIF feature, use PCA, or regularise with Ridge regression.")
else:
    print("All VIFs are below 5 — no severe multicollinearity detected.")

In [ ]:
# Residual diagnostics for the MLR model
y_train_pred = mlr.predict(X_train)
resid_mlr = y_train - y_train_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs Fitted
axes[0].scatter(y_train_pred, resid_mlr, alpha=0.5, edgecolors='white', linewidth=0.3)
axes[0].axhline(0, color='crimson', linewidth=1.5, linestyle='--')
axes[0].set_xlabel("Fitted MPG")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs Fitted", fontweight='bold')

# Histogram
axes[1].hist(resid_mlr, bins=25, edgecolor='white', color='steelblue')
axes[1].axvline(0, color='crimson', linewidth=1.5, linestyle='--')
axes[1].set_xlabel("Residual (MPG)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Residuals", fontweight='bold')

# QQ plot
probplot(resid_mlr, dist="norm", plot=axes[2])
axes[2].set_title("QQ Plot of Residuals", fontweight='bold')

plt.suptitle("MLR Residual Diagnostics — mpg model", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 7. Making Predictions

Once a model is fitted, we can use it to make predictions on new, unseen data.

> **Prediction interval vs Confidence interval:**
>
> - A **confidence interval** for the mean gives the range where the *average* prediction for a given input likely falls.
> - A **prediction interval** for a single new observation is *wider* — it accounts for both the uncertainty in estimating the mean **and** the natural variability of individual data points.
> - In practice, scikit-learn's `predict()` gives a point estimate only. For full intervals, use `statsmodels` which exposes `get_prediction().summary_frame()`.

In [ ]:
# Predict mpg for a specific car configuration
# horsepower=100, weight=2500, cylinders=4, origin=japan
new_car = pd.DataFrame({
    'horsepower': [100],
    'weight':     [2500],
    'cylinders':  [4],
    'origin_japan': [1],
    'origin_usa':   [0]
})

# Ensure column order matches training data
new_car = new_car[X_mpg.columns]

predicted_mpg = mlr.predict(new_car)[0]

print("=== Prediction for a new car ===")
print(f"  Horsepower : 100")
print(f"  Weight     : 2,500 lbs")
print(f"  Cylinders  : 4")
print(f"  Origin     : Japan")
print()
print(f"  Predicted MPG: {predicted_mpg:.2f}")
print()
print(f"  Dataset range — min: {mpg_clean['mpg'].min():.1f}, max: {mpg_clean['mpg'].max():.1f}, mean: {mpg_clean['mpg'].mean():.1f}")
print()
print("Note: This is a point estimate. The actual mpg for this car would likely")
print("fall within a prediction interval of roughly ±3–5 mpg around this value.")

# Actual vs predicted on test set
y_test_pred = mlr.predict(X_test)

plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_test_pred, alpha=0.6, edgecolors='white', linewidth=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         color='crimson', linewidth=2, linestyle='--', label='Perfect prediction')
plt.xlabel("Actual MPG")
plt.ylabel("Predicted MPG")
plt.title(f"Actual vs Predicted MPG — Test Set  (R-sq = {r2_test:.3f})", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---

## 8. Practice Exercises

Work through the exercises below. Try to solve each one before expanding the solution.

---

### Exercise 1

Calculate both **Pearson** and **Spearman** correlations between `size` (party size) and `tip` in the `tips` dataset. Print both r values and p-values, and write 2–3 sentences interpreting the results.

In [ ]:
# Exercise 1 — your code here


<details>
<summary><strong>Solution — Exercise 1</strong> (click to expand)</summary>

```python
# Pearson
r_pe, p_pe = pearsonr(tips['size'], tips['tip'])
# Spearman
r_sp, p_sp = spearmanr(tips['size'], tips['tip'])

print(f"Pearson  r   = {r_pe:.4f}, p = {p_pe:.4f}")
print(f"Spearman rho = {r_sp:.4f}, p = {p_sp:.4f}")

plt.figure(figsize=(7, 4))
sns.boxplot(data=tips, x='size', y='tip')
plt.title(f"Tip by Party Size  (Pearson r={r_pe:.2f}, Spearman rho={r_sp:.2f})")
plt.show()

# Interpretation:
# Both correlations are positive and statistically significant (p < 0.05),
# indicating larger parties tend to leave higher total tips.
# The moderate strength (r around 0.49) suggests party size is one factor
# among several — it does not fully explain tipping behaviour.
# Spearman is close to Pearson, so outlier influence is minimal here.
```

</details>

---

### Exercise 2

Build a **simple linear regression** model predicting `total_bill` from `size`. Print the slope, intercept, and R-squared score. Plot the scatter with the regression line. In plain English, what does the slope tell you?

In [ ]:
# Exercise 2 — your code here


<details>
<summary><strong>Solution — Exercise 2</strong> (click to expand)</summary>

```python
X2 = tips[['size']]
y2 = tips['total_bill']

m2 = LinearRegression().fit(X2, y2)

print(f"Intercept : {m2.intercept_:.4f}")
print(f"Slope     : {m2.coef_[0]:.4f}")
print(f"R-squared : {m2.score(X2, y2):.4f}")

x_rng = np.linspace(tips['size'].min(), tips['size'].max(), 100)
plt.figure(figsize=(8, 5))
plt.scatter(tips['size'], tips['total_bill'], alpha=0.4)
plt.plot(x_rng, m2.coef_[0] * x_rng + m2.intercept_, color='crimson', linewidth=2)
plt.xlabel("Party Size")
plt.ylabel("Total Bill ($)")
plt.title(f"Total Bill ~ Party Size  (R-sq={m2.score(X2,y2):.2f})")
plt.show()

# Plain English:
# The slope means each additional person in the party is associated
# with an average increase of roughly $5 in the total bill.
# R-squared around 0.21 means party size explains only ~21% of the
# variance in bill amount — what people order matters far more.
```

</details>

---

### Exercise 3

Build an **extended MLR model** predicting `tip` from `total_bill`, `size`, and `time` (lunch/dinner). Dummy-encode `time`, fit the model, print all coefficients, and compare R-squared with and without `time`. Does meal time meaningfully improve the model?

In [ ]:
# Exercise 3 — your code here


<details>
<summary><strong>Solution — Exercise 3</strong> (click to expand)</summary>

```python
# Baseline — without time
X_base = tips[['total_bill', 'size']]
y_tip  = tips['tip']
r2_base = LinearRegression().fit(X_base, y_tip).score(X_base, y_tip)

# Encode time (drop_first=True gives time_Lunch: 1=Lunch, 0=Dinner)
time_dummies = pd.get_dummies(tips['time'], prefix='time', drop_first=True)

X_ext = pd.concat([X_base, time_dummies], axis=1)
m3 = LinearRegression().fit(X_ext, y_tip)
r2_ext = m3.score(X_ext, y_tip)

coef_df = pd.DataFrame({'Feature': X_ext.columns, 'Coefficient': m3.coef_})
print(f"Intercept: {m3.intercept_:.4f}")
print(coef_df.to_string(index=False))
print()
print(f"R-squared without time : {r2_base:.4f}")
print(f"R-squared with time    : {r2_ext:.4f}")

# Interpretation:
# Adding 'time' barely changes R-squared, meaning whether it is lunch or dinner
# has very little additional effect on tip amount after controlling for bill and size.
# The coefficient on time_Lunch is small and close to zero, confirming this.
# Conclusion: meal time does not meaningfully improve this model.
```

</details>

---

## Summary

In this tutorial you covered the full workflow for analysing and modelling relationships in data:

| Concept | Key takeaway |
|---|---|
| **Types of relationships** | Linear, non-linear, or none — always look at the scatter plot first |
| **Pearson r** | Measures linear relationship strength; ranges from −1 to +1; needs roughly normal data |
| **Spearman rho** | Rank-based; robust to outliers; use for ordinal or skewed data |
| **Correlation vs causation** | A lurking variable may be driving both measurements |
| **Simple linear regression** | Fits a line; slope = change in y per 1-unit change in x; R-squared = variance explained |
| **Residual analysis** | Three diagnostic plots check linearity, homoscedasticity, and normality |
| **Multiple linear regression** | Multiple predictors; encode categoricals with `pd.get_dummies` before fitting |
| **VIF** | Detects multicollinearity; VIF > 5–10 warrants investigation or remediation |
| **Predictions** | Use `model.predict()`; point estimates only — add a prediction interval for full uncertainty |

---

## Next Steps

- **4.4 Statistical Modelling** — generalised linear models, logistic regression, and model selection criteria
- **5.1 Introduction to ML** — the broader machine learning framework and where regression fits within it
- **Further reading:**
  - *An Introduction to Statistical Learning* (James et al.) — Chapters 2–3 for regression foundations
  - Seaborn `pairplot` documentation for fast exploratory relationship views across many variables
  - `statsmodels.OLS` for full regression summaries including p-values per coefficient and confidence intervals